# 10. 하락봉 직후 3분 호가 +3% 급등 Weighted V6

현재 하락봉의 `last_ask` 대비 다음 3개 연속 분봉 중 `last_bid`가 3% 이상 오른 경우를 양성으로 정의한다. 모델 입력은 기존 OHLC 기반 90개 feature만 사용하며 quote는 미래 라벨 생성에만 사용한다.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'AGENT.md').exists() and (candidate / 'README.md').exists():
            return candidate
    raise FileNotFoundError('프로젝트 루트를 찾지 못했습니다.')

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.quote_surge_binary import run_quote_surge_experiment

assert 'envs/urban' in str(Path(sys.executable).resolve()), sys.executable
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 120)
print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'python: {sys.executable}')

PROJECT_ROOT: /home/user/urbandatalab/YSLee/code/Detecting-entry-signals-for-short-term-trades-right-before-a-rapid-price-surge
python: /home/user/anaconda3/envs/urban/bin/python


## 1. 라벨·dataset 생성과 고정 학습

후보는 `close_t < open_t`인 하락봉으로 한정한다. 현재 및 미래 quote 유효성과 정확한 3분 연속성은 데이터 품질 검사이며 추가 매매 조건은 적용하지 않는다. Train은 세션·종목·3분 bucket당 한 행만 사용하고 날짜별 loss weight를 동일하게 둔다. V6는 balanced positive weight에 1.5배를 적용하고 40 epoch를 학습한다.

In [2]:
result = run_quote_surge_experiment(PROJECT_ROOT)
print(f"device: {result['device']}")
print(f"features: {result['feature_count']}, parameters: {result['parameter_count']:,}")
print(f"positive class weight: {result['positive_weight_base']:.3f} × {result['positive_weight_multiplier']:.2f} = {result['positive_weight']:.3f}")
print(f"exclusions: {result['exclusions']}")
display(result['sampling_balance'])
display(result['history'])

device: cuda
features: 90, parameters: 2,945
positive class weight: 22.787 × 1.50 = 34.180
exclusions: {'non_bearish': 27454, 'invalid_current_quote': 62, 'invalid_future_quote': 196, 'insufficient_future': 598}


,session,candidate_rows,is_train,sampled_rows,positives,sampled_positives
0,session_2026-07-07,1208,True,748,21.0,11.0
1,session_2026-07-08,3209,True,1989,123.0,78.0
2,session_2026-07-09,3049,True,1910,101.0,62.0
3,session_2026-07-10,3181,True,1965,256.0,155.0
4,session_2026-07-13,2802,True,1710,130.0,75.0
5,session_2026-07-14,2230,True,1346,100.0,57.0
6,session_2026-07-15,3195,True,1962,152.0,84.0
7,session_2026-07-16,1984,False,0,91.0,0.0
8,session_2026-07-17,1842,False,0,33.0,0.0


,epoch,weighted_bce
0,1,1.619210
1,2,1.362213
2,3,1.300260
3,4,1.277194
4,5,1.239824
5,6,1.220531
6,7,1.200087
7,8,1.185012
8,9,1.176681
9,10,1.163684


## 2. Train–Test PR-AUC와 날짜 안정성

Class-balanced BCE 때문에 sigmoid 값은 calibration된 확률이 아니라 ranking score다. PR-AUC와 양성률 대비 lift를 우선 해석한다.

In [3]:
display(result['metrics'])

,evaluation_group,session,samples,positives,positive_rate,pr_auc,pr_lift,roc_auc,uncalibrated_logloss,uncalibrated_brier,mean_score
0,train,ALL,18874,883,0.046784,0.219180,4.684950,0.838247,0.645634,0.231131,0.405382
1,test,ALL,3826,124,0.032410,0.114699,3.539030,0.783818,0.530016,0.189396,0.345398
2,train,session_2026-07-07,1208,21,0.017384,0.097054,5.582905,0.879889,0.378680,0.135522,0.252474
3,train,session_2026-07-08,3209,123,0.038330,0.168012,4.383339,0.793552,0.644148,0.236743,0.421997
4,train,session_2026-07-09,3049,101,0.033126,0.220133,6.645395,0.833376,0.489422,0.167185,0.323655
5,train,session_2026-07-10,3181,256,0.080478,0.263053,3.268639,0.815831,0.860401,0.313268,0.509398
6,train,session_2026-07-13,2802,130,0.046395,0.221165,4.766963,0.839918,0.613629,0.214281,0.386456
7,train,session_2026-07-14,2230,100,0.044843,0.198938,4.436323,0.861141,0.646007,0.230162,0.406587
8,train,session_2026-07-15,3195,152,0.047574,0.242144,5.089800,0.826930,0.711112,0.256343,0.436694
9,test,session_2026-07-16,1984,91,0.045867,0.140058,3.053581,0.753296,0.631689,0.229766,0.406599


## 3. Top-K 탐지 성능

각 집합 내부의 상위 1/2/5/10% score가 양성을 얼마나 농축하는지 확인한다. 이는 진단 지표이며 Test quantile로 실전 threshold를 선택하지 않는다.

In [4]:
display(result['top_metrics'])

,evaluation_group,top_fraction,signals,true_positives,precision,recall,minimum_score
0,train,0.01,189,71,0.375661,0.080408,0.954122
1,train,0.02,378,136,0.359788,0.154020,0.926965
2,train,0.05,944,268,0.283898,0.303511,0.871304
3,train,0.10,1888,410,0.217161,0.464326,0.796297
4,test,0.01,39,7,0.179487,0.056452,0.903595
5,test,0.02,77,18,0.233766,0.145161,0.842313
6,test,0.05,192,28,0.145833,0.225806,0.790579
7,test,0.10,383,47,0.122715,0.379032,0.730458


## 4. Train 상위 5% 고정 threshold

Validation이 없으므로 outcome 수익으로 threshold를 최적화하지 않는다. Train score 95% 분위수를 고정하고 Test에 그대로 적용한다.

In [5]:
display(result['threshold'])
display(result['threshold_summary'])
display(result['threshold_sessions'])

,threshold,method,train_score_quantile,uses_train_outcome_for_selection,validation_used
0,0.871242,fixed_train_score_top_5pct,0.95,False,False


,evaluation_group,session,threshold,samples,signals,signal_share,true_positives,precision,recall,actual_positive_clusters,predicted_signal_clusters,captured_positive_clusters
0,train,ALL,0.871242,18874,944,0.050016,268,0.283898,0.303511,640,291,185
1,test,ALL,0.871242,3826,58,0.015159,11,0.189655,0.088710,96,25,8


,evaluation_group,session,threshold,samples,signals,signal_share,true_positives,precision,recall,actual_positive_clusters,predicted_signal_clusters,captured_positive_clusters
0,train,session_2026-07-07,0.871242,1208,6,0.004967,1,0.166667,0.047619,16,4,1
1,train,session_2026-07-08,0.871242,3209,52,0.016204,16,0.307692,0.130081,97,23,12
2,train,session_2026-07-09,0.871242,3049,82,0.026894,25,0.304878,0.247525,74,25,15
3,train,session_2026-07-10,0.871242,3181,352,0.110657,103,0.292614,0.402344,172,100,67
4,train,session_2026-07-13,0.871242,2802,165,0.058887,43,0.260606,0.330769,98,53,35
5,train,session_2026-07-14,0.871242,2230,91,0.040807,27,0.296703,0.270000,71,35,18
6,train,session_2026-07-15,0.871242,3195,196,0.061346,53,0.270408,0.348684,112,51,37
7,test,session_2026-07-16,0.871242,1984,50,0.025202,10,0.200000,0.109890,67,22,7
8,test,session_2026-07-17,0.871242,1842,8,0.004343,1,0.125000,0.030303,29,3,1


## 5. 결론과 제한

이 모델은 TP/SL 수익률 모델이 아니라 3분 급등 이벤트 탐지 모델이다. Test 7/16~17은 과거 실험에서 이미 본 비독립 구간이며, threshold score는 확률로 해석하지 않는다.

In [6]:
data_root = (PROJECT_ROOT / result['config']['data']['root']).resolve()
baseline_version = 'quote_surge_bearish_3m_up3_binary_v5'
baseline_metrics = pd.read_parquet(data_root / 'processed' / f'{baseline_version}_metrics.parquet')
baseline_threshold = pd.read_parquet(data_root / 'processed' / f'{baseline_version}_threshold_summary.parquet')
overall = result['metrics'].query("evaluation_group == 'test' and session == 'ALL'").iloc[0]
selected = result['threshold_summary'].query("evaluation_group == 'test'").iloc[0]
base_train = baseline_metrics.query("evaluation_group == 'train' and session == 'ALL'").iloc[0]
base_test = baseline_metrics.query("evaluation_group == 'test' and session == 'ALL'").iloc[0]
base_selected = baseline_threshold.query("evaluation_group == 'test'").iloc[0]
comparison = pd.DataFrame([
    {'version': 'V5 balanced×1.0 / 15 epoch', 'train_pr_auc': base_train['pr_auc'], 'test_pr_auc': base_test['pr_auc'], 'test_precision': base_selected['precision'], 'test_recall': base_selected['recall']},
    {'version': 'V6 balanced×1.5 / 40 epoch', 'train_pr_auc': result['metrics'].query("evaluation_group == 'train' and session == 'ALL'").iloc[0]['pr_auc'], 'test_pr_auc': overall['pr_auc'], 'test_precision': selected['precision'], 'test_recall': selected['recall']},
])
display(comparison)
display(Markdown(
    f"**Test PR-AUC {overall['pr_auc']:.3f} (base {overall['positive_rate']:.3%}, lift {overall['pr_lift']:.2f}×). "
    f"고정 threshold 신호 {int(selected['signals'])}개 중 양성 적중 {int(selected['true_positives'])}개, "
    f"precision {selected['precision']:.1%}, recall {selected['recall']:.1%}.**"
))

,version,train_pr_auc,test_pr_auc,test_precision,test_recall
0,V5 balanced×1.0 / 15 epoch,0.208719,0.120981,0.241379,0.112903
1,V6 balanced×1.5 / 40 epoch,0.219180,0.114699,0.189655,0.088710


**Test PR-AUC 0.115 (base 3.241%, lift 3.54×). 고정 threshold 신호 58개 중 양성 적중 11개, precision 19.0%, recall 8.9%.**

## 6. 저장 artifact

In [7]:
display(pd.DataFrame({'artifact': result['paths'].keys(), 'path': map(str, result['paths'].values())}))

,artifact,path
0,labels,/home/user/urbandatalab/YSLee/data/stock_data/...
1,tabular,/home/user/urbandatalab/YSLee/data/stock_data/...
2,schema,/home/user/urbandatalab/YSLee/data/stock_data/...
3,predictions,/home/user/urbandatalab/YSLee/data/stock_data/...
4,metrics,/home/user/urbandatalab/YSLee/data/stock_data/...
5,top_metrics,/home/user/urbandatalab/YSLee/data/stock_data/...
6,threshold,/home/user/urbandatalab/YSLee/data/stock_data/...
7,threshold_summary,/home/user/urbandatalab/YSLee/data/stock_data/...
8,threshold_sessions,/home/user/urbandatalab/YSLee/data/stock_data/...
9,sampling_balance,/home/user/urbandatalab/YSLee/data/stock_data/...
